# MiniMax H3 — Colab生成ノートブック（窓際族物語 / colab-video スキル・製品生成専用）

動画スキルで制作済みのバンドル（キーフレーム＋wav＋`ch*_workflow.json`＋`h3_run.py`）を、
ColabのL4 GPUでチャプター毎に動画化する。配管検証は2026-08に完了済みのため、このノートブックは生成だけに特化している。

使い方: **セル1だけ編集**して、あとは上から順に実行（1→9）。パイロット（セリフ有りチャプター）を先に1本生成して確認してから残りを回すこと。
セル10（アドホック生成）は、チャプター定義に縛られず素材＋プロンプトから単発で1本作るときに使う。

課金の注意: CUは「GPUランタイム接続中の時間」で消費される（セル実行中でなくても）。終わったら必ず「ランタイム → ランタイムを接続解除して削除」。


In [ ]:
#@title 1. 設定（毎セッションここだけ編集）
CHAPTERS = []                # 生成するチャプター。まずパイロット（セリフ有り）1本 → 合格後に残り。例: ["ch2"] → ["ch1", "ch3"]
BUNDLE_ZIP_FROM_DRIVE = ""   # 例 "/content/drive/MyDrive/41_okayaman_bundle.zip"。空ならセル5でブラウザからアップロード
WEIGHTS_DRIVE_DIR = ""       # 例 "/content/drive/MyDrive/h3_weights"。Drive上の重みをsymlinkで直接使い（ディスク消費ゼロ）、無い分は保存する
SAVE_WEIGHTS_TO_DRIVE = True # WEIGHTS_DRIVE_DIR設定時、HFから落とした重みをDriveへ保存する（次回セッションが数分で立ち上がる）
OUT_DRIVE_DIR = ""           # 例 "/content/drive/MyDrive/h3_outputs"。設定すると各チャプター完了ごとに即Driveへ退避（切断事故に強い）
NEED_I2V = True              # I2Vチャプター（fl2va 21GB）を使う
NEED_R2V = True              # R2Vチャプター（ref2va 21GB）を使う
COMFY_FLAGS = []             # 生成中に CUDA out of memory が出たら ["--lowvram"] にしてセル7から再実行
print("OK:", dict(CHAPTERS=CHAPTERS, NEED_I2V=NEED_I2V, NEED_R2V=NEED_R2V,
                  WEIGHTS_DRIVE_DIR=WEIGHTS_DRIVE_DIR or "(未使用)", OUT_DRIVE_DIR=OUT_DRIVE_DIR or "(未使用)"))


In [ ]:
#@title 2. 環境チェック（GPU世代→重みバリアント自動選択。CPUランタイムは重みのDrive配置専用モード）
import shutil, torch, psutil
if torch.cuda.is_available():
    NAME, CAP = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
    assert CAP >= (8, 0), f"{NAME} は生成に使えない（Turing以下）。L4以上のGPUを選ぶ"
else:
    # 無料CPUランタイム: 生成はできないが、セル3→4で重みをDL→Driveへ配置する用途（0円）に使える
    NAME, CAP, VRAM = "CPU（重み配置専用モード — セル4まで実行、生成セルは不可）", (8, 9), 0.0
    print("⚠ GPUなし: 重みのDrive配置専用モードとして続行（L4/Ada向けバリアントを選択）")
RAM = psutil.virtual_memory().total / 2**30
DISK = shutil.disk_usage("/content").free / 2**30
print(f"GPU: {NAME} (SM {CAP[0]}.{CAP[1]})  VRAM {VRAM:.1f} GiB  RAM {RAM:.1f} GiB  空きディスク {DISK:.1f} GiB")

if CAP >= (10, 0):        # Blackwell
    ENCODER, FP8 = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", True
elif CAP >= (8, 9):       # Ada (L4/RTX40xx) / Hopper
    ENCODER, FP8 = "qwen3vl_32b_minimax_h3_int8_convrot.safetensors", True
else:                     # Ampere (A100)
    ENCODER, FP8 = "qwen3vl_32b_minimax_h3_int8_convrot.safetensors", False
Q = "fp8_scaled" if FP8 else "int8_convrot"
UNET_I2V = f"minimax_h3_fl2va_pruned_{Q}.safetensors"
UNET_R2V = f"minimax_h3_ref2va_pruned_{Q}.safetensors"
print("重み:", UNET_I2V, "/", UNET_R2V, "/", ENCODER)

# ディスク見積り: 実測でL4のディスクは65GB程度しかないことがある。両モードの実体DL（計約75GB）は
# 収まらないので、その場合は WEIGHTS_DRIVE_DIR（symlink直参照）を使うか、NEED_I2V/R2Vを片方ずつにする
est = 6 + (27 if not WEIGHTS_DRIVE_DIR else 0) + 21 * ((NEED_I2V + NEED_R2V) if not WEIGHTS_DRIVE_DIR else 0)
if not WEIGHTS_DRIVE_DIR and DISK < est + 8:
    print(f"⚠ 空き{DISK:.0f}GiBに対しDL見積り約{est}GB — WEIGHTS_DRIVE_DIRの利用か、NEEDフラグを片方ずつにすることを推奨")


In [ ]:
%%bash
# 3. ComfyUIインストール＋aria2導入（2〜3分）
set -e
apt-get -yq install aria2 > /dev/null 2>&1 || true
cd /content
if [ ! -d ComfyUI ]; then
  git clone --depth 1 https://github.com/comfyanonymous/ComfyUI
fi
cd ComfyUI
pip install -q -r requirements.txt
test -f comfy_extras/nodes_minimax_h3.py && echo "MiniMax H3 nodes: OK" \
  || { echo "ERROR: nodes_minimax_h3.py が無い — ComfyUIが古い"; exit 1; }


In [ ]:
#@title 4. 重み配置（Drive優先 → 無い分はaria2でDL。レジューム対応・進捗表示あり）
import os, shutil, subprocess
REPO = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main"
SUB = lambda n: "vae" if "_vae_" in n else ("text_encoders" if n.startswith("qwen3vl") else "diffusion_models")
MIN_BYTES = {  # 不完全ファイル検出用の下限（実サイズの少し下）
    "minimax_h3_video_vae_fp16.safetensors": 5_000_000_000,
    "minimax_h3_audio_vae_fp32.safetensors": 550_000_000,
    "qwen3vl_32b_minimax_h3_int8_convrot.safetensors": 26_000_000_000,
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 15_000_000_000,
    "minimax_h3_fl2va_pruned_fp8_scaled.safetensors": 20_900_000_000,
    "minimax_h3_ref2va_pruned_fp8_scaled.safetensors": 20_900_000_000,
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors": 20_000_000_000,
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors": 20_000_000_000,
}
need = ["minimax_h3_video_vae_fp16.safetensors", "minimax_h3_audio_vae_fp32.safetensors", ENCODER]
if NEED_I2V: need.append(UNET_I2V)
if NEED_R2V: need.append(UNET_R2V)

if WEIGHTS_DRIVE_DIR or BUNDLE_ZIP_FROM_DRIVE or OUT_DRIVE_DIR:
    from google.colab import drive as _gd
    _gd.mount("/content/drive")
drive_dir = WEIGHTS_DRIVE_DIR or None
if drive_dir:
    os.makedirs(drive_dir, exist_ok=True)

def ok_size(path, n):
    return os.path.exists(path) and os.path.getsize(path) >= MIN_BYTES.get(n, 1)

for n in need:
    dst = f"/content/ComfyUI/models/{SUB(n)}/{n}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if ok_size(dst, n):
        print("skip（配置済み）", n)
        continue
    drv = f"{drive_dir}/{n}" if drive_dir else None
    if drv and ok_size(drv, n):
        if os.path.lexists(dst):
            os.remove(dst)
        os.symlink(drv, dst)  # Drive実体を直参照: ローカルディスク消費ゼロ（初回ロードはFUSE越しでやや遅い）
        print("drive OK（symlink）", n)
        continue
    print(f"=== {n} をaria2でDL（16並列・15秒毎に進捗表示・中断してもレジューム可） ===", flush=True)
    r = subprocess.run(["aria2c", "-c", "-x16", "-s16", "--file-allocation=none",
                        "--summary-interval=15", "--console-log-level=warn",
                        "-d", os.path.dirname(dst), "-o", n, f"{REPO}/{SUB(n)}/{n}"])
    assert r.returncode == 0 and ok_size(dst, n), f"{n} のDLに失敗 — このセルを再実行すれば途中から再開する"
    print("hf OK", n)
    if drv and SAVE_WEIGHTS_TO_DRIVE:
        try:
            print(f"  -> Driveへ保存中（FUSE越しで時間がかかる。次回以降の高速化用）: {drv}", flush=True)
            shutil.copy(dst, drv)
        except OSError as e:  # Drive容量・一時キャッシュ枯渇などでも生成は止めない
            print(f"  ⚠ Drive保存に失敗（生成には影響なし。後で無料CPUセッションでの配置を推奨）: {e}")
            if os.path.exists(drv):
                os.remove(drv)
print(f"完了。空きディスク: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")


In [ ]:
#@title 5. バンドル投入（zipをアップロード or Driveから）→ ComfyUI/input/ へ配備
import glob, os, shutil, zipfile

if BUNDLE_ZIP_FROM_DRIVE:
    zp = BUNDLE_ZIP_FROM_DRIVE
    assert os.path.exists(zp), f"{zp} が無い"
else:
    from google.colab import files
    print("バンドルzip（<NN>_<slug>_bundle.zip）を選択:")
    up = files.upload()
    zp = "/content/" + next(iter(up))

shutil.rmtree("/content/bundle", ignore_errors=True)
zipfile.ZipFile(zp).extractall("/content/bundle")
hits = glob.glob("/content/bundle/**/script.md", recursive=True)
assert hits, "zip内にscript.mdが見つからない — ラン専用ディレクトリごとzipしたか確認"
BUNDLE = os.path.dirname(hits[0])
print("BUNDLE =", BUNDLE)

inp = "/content/ComfyUI/input"
os.makedirs(inp, exist_ok=True)
n = 0
for p in sorted(glob.glob(f"{BUNDLE}/*.png") + glob.glob(f"{BUNDLE}/*.wav")):
    if os.path.basename(p).startswith("ref_canvas_"):
        continue
    shutil.copy(p, inp)
    n += 1
print(f"{n} files -> ComfyUI/input/")


In [ ]:
#@title 6. workflowをこのGPUの重み名に調整（SaveVideoのcodec補完込み）
import glob, json, os
wfs = sorted(glob.glob(f"{BUNDLE}/ch*_workflow.json"))
assert wfs, f"{BUNDLE} に ch*_workflow.json が無い — バンドル作成時にworkflowを生成したか確認"
for wf in wfs:
    with open(wf) as f:
        d = json.load(f)
    for node in d.values():
        ins = node.get("inputs", {})
        for k, v in ins.items():
            if not isinstance(v, str):
                continue
            if v.startswith("minimax_h3_fl2va"):
                ins[k] = UNET_I2V
            elif v.startswith("minimax_h3_ref2va"):
                ins[k] = UNET_R2V
            elif v.startswith("qwen3vl_32b"):
                ins[k] = ENCODER
        if node.get("class_type") == "SaveVideo":  # ComfyUI新版(2026-08〜)はcodec必須
            ins.setdefault("codec", "auto")
            ins.setdefault("format", "auto")
    with open(wf, "w") as f:
        json.dump(d, f, indent=1)
    print("adjusted", os.path.basename(wf))
print("重み:", UNET_I2V, "/", UNET_R2V, "/", ENCODER)


In [ ]:
#@title 7. ComfyUI起動（プロセスが落ちたらこのセルを再実行）
import json, subprocess, sys, time, urllib.request
SERVER = "127.0.0.1:8188"
subprocess.run(["pkill", "-f", "main.py --listen"], check=False)
time.sleep(2)
LOG = open("/content/comfyui.log", "w")
PROC = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", *COMFY_FLAGS],
    cwd="/content/ComfyUI", stdout=LOG, stderr=subprocess.STDOUT)
INFO = None
for _ in range(90):
    try:
        INFO = json.load(urllib.request.urlopen(f"http://{SERVER}/object_info", timeout=5))
        break
    except Exception:
        time.sleep(2)
assert INFO, "ComfyUIが起動しない — !tail -50 /content/comfyui.log で確認"
h3_nodes = sorted(k for k in INFO if k.startswith("MiniMaxH3"))
assert h3_nodes, "H3ノードが登録されていない — ComfyUIのバージョンを確認"
print("MiniMaxH3 nodes:", h3_nodes)


In [ ]:
#@title 8. チャプター生成（完了ごとに即退避。生成済みはスキップ＝中断・再開に強い）
import os, shutil, subprocess, sys
assert CHAPTERS, "セル1の CHAPTERS を設定（まずパイロット1本 → 合格後に残り）"
os.makedirs("/content/outputs", exist_ok=True)
if OUT_DRIVE_DIR:
    os.makedirs(OUT_DRIVE_DIR, exist_ok=True)
for ch in CHAPTERS:
    wf = os.path.join(BUNDLE, f"{ch}_workflow.json")
    assert os.path.exists(wf), f"{wf} が無い"
    out = f"/content/outputs/{ch}.mp4"
    if os.path.exists(out):
        print("skip（生成済み）", ch)
        continue
    print(f"=== {ch} 生成開始（動画1秒あたり約4.5分@L4。ポーリング出力が続いていれば正常）===", flush=True)
    r = subprocess.run([sys.executable, os.path.join(BUNDLE, "h3_run.py"), wf, "--out", out])
    if r.returncode != 0:
        raise RuntimeError(f"{ch} が失敗 — !tail -80 /content/comfyui.log で確認。成功済み分はセル9で回収できる")
    if OUT_DRIVE_DIR:
        shutil.copy(out, OUT_DRIVE_DIR)
        print(f"  -> Drive退避済み: {OUT_DRIVE_DIR}/{ch}.mp4")
print("指定チャプター完了。ブラウザにも落とすならセル9へ")


In [ ]:
#@title 9. 成果物の回収（zip→ブラウザDL。Drive退避済みならスキップ可）
import glob, subprocess
outs = sorted(glob.glob("/content/outputs/*.mp4"))
assert outs, "/content/outputs にmp4が無い"
print(*outs, sep="\n")
subprocess.run(["zip", "-j", "-q", "/content/h3_outputs.zip", *outs], check=True)
from google.colab import files
files.download("/content/h3_outputs.zip")
print("回収したら「ランタイム → ランタイムを接続解除して削除」で課金を止めること")


In [ ]:
#@title 10.（任意）アドホック生成 — 画像・音声・プロンプトを直接指定して1本作る
# チャプター定義に縛られない単発生成。素材はバンドル同梱ファイル名で指定（新素材は左のファイルペインで
# /content/ComfyUI/input/ へドラッグ＆ドロップしてから指定）。framesは17k+5グリッド（90,124,141,158,...）。
# H3の埋め込み音声は入力wavと同等（2026-08実測）なので、出力の音声はそのまま最終成果物に使える。
ADHOC = dict(
    mode="r2v",      # "i2v"=開始/終了フレーム固定・音声なし / "r2v"=参照画像(≦9)+音声(≦3・各2〜15s)・リップシンク
    frames=124,
    prompt="Required attached input files: <Picture 1> = XXX.png — ...; <Audio 1> = YYY.wav — spoken line, use AS-IS. "
           "The video starts EXACTLY on <Picture 1>. ... (S1) speaks — he says <d>[Japanese] セリフ</d>, lip-syncing to <Audio 1>. "
           "Soundscape: ... Music: no background music.",
    first="chN_start.png", last="chN_end.png",  # i2vのみ
    images=["XXX.png"], audio=["YYY.wav"],      # r2vのみ（<Picture N>/<Audio N>の接続順）
    out="adhoc1",
)
import json, os, subprocess, sys
pf = os.path.join(BUNDLE, f"{ADHOC['out']}_prompt.txt")
with open(pf, "w") as f:
    f.write(ADHOC["prompt"])
wf = os.path.join(BUNDLE, f"{ADHOC['out']}_workflow.json")
cmd = [sys.executable, os.path.join(BUNDLE, "build_h3_workflow.py"), "--mode", ADHOC["mode"],
       "--out", wf, "--prompt-file", pf, "--frames", str(ADHOC["frames"]),
       "--prefix", f"video/{ADHOC['out']}",
       "--encoder", ENCODER, "--unet-i2v", UNET_I2V, "--unet-r2v", UNET_R2V]
if ADHOC["mode"] == "i2v":
    cmd += ["--first", ADHOC["first"], "--last", ADHOC["last"]]
else:
    for im in ADHOC["images"]:
        cmd += ["--image", im]
    for au in ADHOC["audio"]:
        cmd += ["--audio", au]
subprocess.run(cmd, check=True)
with open(wf) as f:  # 旧builder対策のcodec補完
    _d = json.load(f)
for _n in _d.values():
    if _n.get("class_type") == "SaveVideo":
        _n["inputs"].setdefault("codec", "auto")
        _n["inputs"].setdefault("format", "auto")
with open(wf, "w") as f:
    json.dump(_d, f, indent=1)
os.makedirs("/content/outputs", exist_ok=True)
r = subprocess.run([sys.executable, os.path.join(BUNDLE, "h3_run.py"), wf,
                    "--out", f"/content/outputs/{ADHOC['out']}.mp4"])
print("結果:", "完了 -> セル9で回収" if r.returncode == 0 else "失敗 — !tail -80 /content/comfyui.log")


## 後工程（ローカル）

回収した`chN.mp4`をラン専用ディレクトリに置き、ffmpegでconcat結合する（埋め込み音声をそのまま使う）。
エンドカード等の文字入れはCapCutで後付け（H3には文字を描かせない）。手順の詳細はランの`H3_COLAB.md`／
`.claude/skills/colab-video/SKILL.md`を参照。

トラブル時はセル出力と `!tail -80 /content/comfyui.log` をClaude Code / Cursorに貼れば診断できる。
